# AKANE E2-12：つながりを、いろいろなAIに付けさせて比べる

E2-11では、主張どうしのつながりを**くじでわざと**反対にした。今回は、つながりを**実際のAI 4つ**に付けさせる。
外れは作り物ではなく、そのAIが本当にやる外れになる。

- つながりを付けるAI：NLI専用の小・中・大（DeBERTa系）と、Gemma-2 2Bに聞く形
- 主張・木・自信・正解はE2-10と同じ（BTPropの公開データ、FactCheckGPT と WikiBio）
- 比べる：AIの自信／BTProp（つながりを全部信じる）／ルールを全部信じるBP／ルールも疑うBP

計画は sims/e2_btprop/PLAN_E2-12.md（結果を見る前に固定）。
**時間の目安**（T4 GPU）：つながりの判定 15〜25分、計算 5〜10分。

## 0. 準備
1. 「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU**。
2. 鍵マークの `HF_TOKEN`（前と同じもの）にアクセスを許可。
3. 「ランタイム」→「すべてのセルを実行」。途中で止まったら、もう一度「すべてのセルを実行」で続きから動く（判定は保存される）。

In [ ]:
# ===== 設定 =====
USE_MOCK_NLI  = False   # True: 作り物の判定（動作確認用。結果は研究に使わない）
SAVE_TO_DRIVE = True
NLI_MODELS = {   # 名前: Hugging Faceの名前
    "NLI小(DeBERTa-xsmall)": "cross-encoder/nli-deberta-v3-xsmall",
    "NLI中(DeBERTa-base)":   "cross-encoder/nli-deberta-v3-base",
    "NLI大(DeBERTa-large)":  "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli",
}
GEMMA_ID = "google/gemma-2-2b-it"

In [ ]:
import os, json, time, subprocess, collections
import numpy as np
OUT = "/content/E2_results"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/AKANE_アーキテクチャ/E2_results"
os.makedirs(OUT, exist_ok=True)
LABELS_JSON = f"{OUT}/E2-10_wikibio_labels.json"     # E2-10で保存したWikiBioの正解
assert os.path.exists(LABELS_JSON), "E2_results に E2-10_wikibio_labels.json が見つからない"
# BTProp（公開の木・自信・コード）と FactCheckGPT の正解を取ってくる
if not os.path.exists("/content/BTProp"):
    subprocess.run("git clone -q -b master https://github.com/UCSB-NLP-Chang/BTProp.git /content/BTProp", shell=True, check=True)
os.chdir("/content/BTProp"); os.makedirs("eval_dataset", exist_ok=True)
if not os.path.exists("eval_dataset/Factbench.jsonl"):
    subprocess.run("wget -q -O eval_dataset/Factbench.jsonl https://raw.githubusercontent.com/yuxiaw/Factcheck-GPT/main/Factbench.jsonl", shell=True, check=True)
!pip -q install pyyaml scikit-learn
print("保存先:", OUT)

## 1. WikiBioの正解を、BTPropの文の順に合わせる（E2-10と同じ手順、1分弱）

In [ ]:
%%writefile /content/e212.py
"""E2-12: つながり（NLI）を、別のAIが付けたものに替えて、BTProp と ルールを疑うBP を比べる。
cwd=BTPropのclone（eval_dataset/ に Factbench.jsonl・wikibio_preprocess_v2.json）。
python e212.py factcheckgpt|wikibio E2-12_nli.json OUT.json"""
import json, sys, itertools, collections
import numpy as np
from sklearn.metrics import roc_auc_score
sys.path.insert(0, '.')
from src.hmm import HMMTreeNode
from src import data_utils as du

ds, nlip, outp = sys.argv[1], sys.argv[2], sys.argv[3]
NLI = json.load(open(nlip))
NAMES = ['entail', 'neutral', 'contradict']
SOURCES = ['元のNLI'] + list(NLI['models'])
TAB = {}
for m, v in NLI['models'].items():
    TAB[m] = {(p, c): f"{NAMES[int(np.argmax(f))]}-{NAMES[int(np.argmax(b))]}"
              for (p, c), f, b in zip(NLI['pairs'], v['fwd'], v['bwd'])}
SRC = {'name': '元のNLI'}
MISS = [0]
def relof(rel, cur, p):
    if SRC['name'] == '元のNLI': return rel
    return TAB[SRC['name']].get((cur, p), rel)
if ds == 'factcheckgpt':
    data = du.load_factcheckgpt_processed_human()
    lab = lambda l: None if isinstance(l, str) else int(not l)       # 1=嘘
else:
    data = du.load_wikibio_processed_human()[120:]
    lab = lambda l: 1 if 'inaccurate' in l else 0
conf = {c['statement']: c['probability'] for c in json.load(open(f'logs/conf_estimation/{ds}_chatgpt.json'))}
prem = {d['statement']: d for d in json.load(open(f'logs/belief_trees/{ds}_chatgpt.json'))}

def build(s):  # BTPropの hmm_forward.py と同じ手順で木を組む
    root = HMMTreeNode('configs/hmm.yaml', s, conf.get(s, 0.5))
    if s not in prem: return None
    seen, todo = [], [s]
    while todo:
        cur = todo.pop()
        if cur not in prem or cur in seen: continue
        seen.append(cur)
        for cd in prem[cur]['child_nodes']:
            kids, rels, m = cd['child_nodes'], cd['mnli_relations'], cd['prem_gen_method']
            if m not in ('decompose', 'logical relation'): kids, rels = kids[:5], rels[:5]
            for i, p in enumerate(kids):
                if p == cur: continue
                rel = rels[i] if m == 'decompose' else relof(rels[i], cur, p)
                if rel == 'neutral-neutral': continue
                if m != 'perturbation' and p not in conf: MISS[0] += 1; continue   # 元で捨てられ自信の無い子
                pc = 1.0 if m == 'perturbation' else conf[p]
                L = root.add_node_via_statement(cur, p, pc, rel, m)
                if L < 2 and m != 'perturbation': todo.append(p)
    return root

def make_items():
    items = []   # (回答番号, ラベル, 自信, BTProp, 木)
    for ri, ex in enumerate(data):
        for s, l in zip(ex['segmented_response'], ex['labels']):
            y = lab(l)
            if y is None or s not in conf: continue
            t = build(s)
            bt = (t.get_posterior() if (t and t.child_nodes) else (1.0 if t is None else conf[s]))
            items.append((ri, y, conf[s], bt, t))
    return items
items = make_items()
print(len(items), 'lies', sum(i[1] for i in items), 'with tree', sum(1 for i in items if i[4] and i[4].child_nodes))

E_T = np.array(items[0][4].emission_probs_true if items[0][4] else HMMTreeNode('configs/hmm.yaml','x',.5).emission_probs_true)
H0 = HMMTreeNode('configs/hmm.yaml', 'x', .5)
def bt_field(c):  # BTPropの表を場に
    return float(np.log(H0.get_p_observation_given_hidden(True, c) / H0.get_p_observation_given_hidden(False, c)))
lgt = lambda c: float(np.log(np.clip(c, 1e-4, 1-1e-4) / (1 - np.clip(c, 1e-4, 1-1e-4))))

def lse(a, b): return np.logaddexp(a, b)
def msg(node, P, pert=False):
    """log m(z), z=0,1：この節と下の木の証拠"""
    if pert: h = P['b']
    elif P.get('bt'): h = bt_field(node.prior_confidence)
    else: h = P['lam'] * (lgt(node.prior_confidence) - P['shift']) * P['scale']
    m = np.array([0.0, h]); W = P['W']
    dec = [c for c, g in zip(node.child_nodes, node.general_relations) if g == 'decompose']
    for c, rel, g in zip(node.child_nodes, node.nli_relations, node.general_relations):
        if g == 'decompose': continue
        mc = msg(c, P, pert=(g == 'perturbation'))
        pen = np.zeros((2, 2))   # [zp, zc]
        if rel == 'entail-entail': pen[1, 0] = pen[0, 1] = W
        elif rel == 'entail-neutral': pen[1, 0] = W
        elif rel == 'neutral-entail': pen[0, 1] = W
        elif 'contradict' in rel: pen[1, 1] = W
        m = m + np.array([lse(mc[0]-pen[zp, 0], mc[1]-pen[zp, 1]) for zp in (0, 1)])
    if dec:
        ms = [msg(c, P) for c in dec]
        A = sum(x[1] for x in ms); T = sum(lse(x[0], x[1]) for x in ms)
        notA = T + np.log(max(1 - np.exp(A - T), 1e-300))
        m = m + np.array([lse(notA, A - W), lse(A, notA - W)])
    return m - m.max()

def score(it, P):   # 嘘らしさ
    ri, y, c, bt, t = it
    if t is None:  # 木がない主張は自分の自信だけ（BTPropの公開コードは確率1にしている）
        h = bt_field(c) if P.get('bt') else P['lam'] * (lgt(c) - P['shift']) * P['scale']
        return 1 - 1 / (1 + np.exp(-h))
    m = msg(t, P); p = 1 / (1 + np.exp(m[0] - m[1]))
    return 1 - p

Y = np.array([i[1] for i in items]); R = np.array([i[0] for i in items])
C = np.array([i[2] for i in items]); BT = np.array([i[3] for i in items])
L = np.array([lgt(c) for c in C])
BTf = np.where([i[4] is None for i in items], C, BT)
print('no tree', sum(i[4] is None for i in items))
resp = np.unique(R); res = []
for src, sp in itertools.product(SOURCES, range(5)):
    SRC['name'] = src; items = make_items()
    BT = np.array([i[3] for i in items])
    rng = np.random.default_rng(sp)
    calr = set(rng.choice(resp, int(round(0.2 * len(resp))), replace=False))
    cal = np.array([R[k] in calr for k in range(len(items))]); test = ~cal
    scale = 2.0 / L[cal].std()
    shift = max(np.quantile(L[cal], np.linspace(.02, .98, 97)), key=lambda b: ((L[cal] > b) == (Y[cal] == 0)).mean())
    def auc(sc, msk): return float(roc_auc_score(Y[msk], np.asarray(sc)[msk]))
    def tune(grid):
        best = None
        for P in grid:
            P = dict(P, shift=shift, scale=scale)
            a = auc([score(items[k], P) if cal[k] else 0 for k in range(len(items))], cal)
            if best is None or a > best[0]: best = (a, P)
        return best[1]
    G = lambda Ws: [dict(lam=l, W=w, b=b) for l, w, b in itertools.product([.5, 1, 2], Ws, [0, .5, 1])]
    Ps = {'疑うBP': tune(G([.5, 1, 2, 4])), '全部信じるBP': tune(G([20])),
          '疑うBP_BTPropの表': tune([dict(bt=True, W=w, b=b, lam=1) for w, b in itertools.product([.5, 1, 2, 4, 20], [0, .5, 1, 2])])}
    row = {'つながり': src, 'split': sp, 'n_test': int(test.sum()), 'lies_test': int(Y[test].sum()),
           'AIの自信': auc(1 - C, test), 'BTProp': auc(1 - BT, test), 'BTProp_木なしは自信': auc(1 - np.where([i[4] is None for i in items], C, BT), test)}
    for k, P in Ps.items():
        row[k] = auc([score(it, P) for it in items], test); row[k + '_設定'] = {a: P[a] for a in P if a not in ('shift', 'scale')}
    print(row); res.append(row)
json.dump(res, open(outp, 'w'), ensure_ascii=False, indent=1)

# まとめ（5通りの平均）
keys = ['AIの自信', 'BTProp', 'BTProp_木なしは自信', '全部信じるBP', '疑うBP']
print('\n平均AUROC', ds)
for src in SOURCES:
    rs = [r for r in res if r['つながり'] == src]
    bt = 'BTProp_木なしは自信' if ds == 'factcheckgpt' else 'BTProp'
    d = [r['疑うBP'] - r[bt] for r in rs]
    print(f"{src:28s}", ' '.join(f"{k}={np.mean([r[k] for r in rs]):.3f}" for k in keys),
          f"| 疑う−BTProp {np.mean(d):+.3f} ({sum(x > 0 for x in d)}/5)")


In [ ]:
"""BTPropのWikiBio前処理（代名詞を名前に置き換えた文、非公開）に、公開の正解ラベルを順番で合わせる。
公開の hmm_posterior.json（評価文の順）と、生の gpt3_sentences（121人目以降、126番除く）を文の似かたで並べる。
cwd=clone。出力 eval_dataset/wikibio_preprocess_v2.json（BTPropの形）"""
import json, difflib, numpy as np
raw = json.load(open(LABELS_JSON))
raw = [raw[i] for i in range(len(raw)) if i != 126][120:]
A = [(k, j, s, r['annotation'][j]) for k, r in enumerate(raw) for j, s in enumerate(r['gpt3_sentences'])]
B = [x['statement'] for x in json.load(open('logs/conf_estimation/hmm_posterior.json'))]
sim = lambda a, b: difflib.SequenceMatcher(None, a, b).ratio()
n, m = len(A), len(B); G = -0.3
S = np.full((n + 1, m + 1), -1e9); S[0, :] = np.arange(m + 1) * G; S[:, 0] = np.arange(n + 1) * G
bk = np.zeros((n + 1, m + 1), int)
band = 60
for i in range(1, n + 1):
    for j in range(max(1, i - band), min(m, i + band) + 1):
        c = [S[i-1, j-1] + sim(A[i-1][2], B[j-1]) - 0.5, S[i-1, j] + G, S[i, j-1] + G]
        bk[i, j] = int(np.argmax(c)); S[i, j] = max(c)
i, j, pairs = n, m, []
while i > 0 and j > 0:
    if bk[i, j] == 0: pairs.append((i-1, j-1, sim(A[i-1][2], B[j-1]))); i -= 1; j -= 1
    elif bk[i, j] == 1: i -= 1
    else: j -= 1
pairs = pairs[::-1]
sims = np.array([p[2] for p in pairs])
print('raw', n, 'btprop', m, 'paired', len(pairs), 'sim<0.5', int((sims < .5).sum()), 'median', float(np.median(sims)))
# 評価で使う形：1人＝1回答。文はBTPropの文、ラベルは生の注釈
by = {}
for a, b, s in pairs:
    if s < 0.5: continue
    k = A[a][0]; by.setdefault(k, ([], []))
    by[k][0].append(B[b]); by[k][1].append(A[a][3])
data = [{'segmented_response': v[0], 'labels': v[1]} for k, v in sorted(by.items())]
# BTPropの読み込み（126番を捨てて[120:]）でちょうど data になるよう空きを入れる
L = [{'segmented_response': [], 'labels': []}] * 120 + data
L.insert(126, {'segmented_response': [], 'labels': []})   # data_utils が126番を捨てる分
json.dump(L, open('eval_dataset/wikibio_preprocess_v2.json', 'w'), ensure_ascii=False)
print('passages', len(data), 'sentences', sum(len(d['labels']) for d in data))
low = [(A[a][2][:70], B[b][:70]) for a, b, s in pairs if s < .6][:6]
for x in low: print(x)

## 2. つながりを判定する組を集める（評価に届く組＋聞き方選び用の200組）

In [ ]:
"""E2-12 判定する組を集める。cwd=BTPropのclone（logs/ と eval_dataset/ があること）。
PAIRS[ds] = [(親, 子)]（分解以外の枝、BTPropと同じく作り変えは5本まで）
ORIG[ds][(親,子)] = 元のNLI（'entail-neutral' の形。前＝親→子、後＝子→親）
HELD = 聞き方選び用の200組（WikiBioの、評価に出る主張から届かない木＝前120人の分）"""
import json, random
TREES = {ds: {d['statement']: d for d in json.load(open(f'logs/belief_trees/{ds}_chatgpt.json'))}
         for ds in ('factcheckgpt', 'wikibio')}

def edges(prem, s):
    for cd in prem[s]['child_nodes']:
        kids, rels, m = cd['child_nodes'], cd['mnli_relations'], cd['prem_gen_method']
        if m not in ('decompose', 'logical relation'): kids, rels = kids[:5], rels[:5]
        for p, r in zip(kids, rels):
            if p != s: yield p, r, m

PAIRS, ORIG = {}, {}
for ds, prem in TREES.items():
    o = {}
    for s in prem:
        for p, r, m in edges(prem, s):
            if m != 'decompose': o.setdefault((s, p), r)
    ORIG[ds] = o; PAIRS[ds] = sorted(o)

def reach(prem, roots):
    seen, todo = set(), list(roots)
    while todo:
        s = todo.pop()
        if s in seen or s not in prem: continue
        seen.add(s)
        todo += [p for p, r, m in edges(prem, s) if m != 'perturbation']
    return seen

def eval_statements(ds):
    if ds == 'factcheckgpt':
        out = []
        for line in open('eval_dataset/Factbench.jsonl', encoding='utf-8'):
            if line.strip():
                d = json.loads(line)
                if d['source'] == 'factcheckgpt': out += d['claims']
        return out
    return [x['statement'] for x in json.load(open('logs/conf_estimation/hmm_posterior.json'))]

_used = reach(TREES['wikibio'], eval_statements('wikibio'))
_cand = [pq for pq in PAIRS['wikibio'] if pq[0] not in _used]
HELD = random.Random(0).sample(_cand, 200)
HELD_SET = set(HELD)
# 実際に聞く組：評価に届く組（FactCheckGPTは全部）＋聞き方選びの200組
RUN = sorted(set(PAIRS['factcheckgpt']) | {pq for pq in PAIRS['wikibio'] if pq[0] in _used} | HELD_SET)


print('実際に聞く組', len(RUN), '／聞き方選び', len(HELD))

## 3. つながりを判定する
各組を両方向（親→子、子→親）で「含まれる／どちらとも言えない／矛盾する」の3択に。
Gemmaは聞き方を3つ用意し、**評価に使わない200組**で元の判定と一番合うものを1回だけ選んでから、全部に使う。

In [ ]:
# E2-12 つながりの判定（Colabのセル）。RUN・HELD・ORIG はセル2で作ったもの。
# 結果: E2-12_nli.json = {"pairs": [[親,子]], "models": {名前: {"fwd": [[含,どちらとも,矛]], "bwd": [...]}}, "gemma_prompt": {...}}
if not USE_MOCK_NLI:
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
NAMES = ["entail", "neutral", "contradict"]
nli_json = f"{OUT}/E2-12_nli.json"
res = json.load(open(nli_json)) if os.path.exists(nli_json) else {"pairs": [list(x) for x in RUN], "models": {}}
assert res["pairs"] == [list(x) for x in RUN], "保存済みの組と合わない。E2-12_nli.json を消してやり直す"
def save(): json.dump(res, open(nli_json, "w"), ensure_ascii=False); print("  保存:", nli_json, flush=True)

def mock(pairs, seed, acc):   # 作り物の判定：元の判定を acc の割合で当て、残りはでたらめ
    r = np.random.default_rng(seed); F, B = [], []
    for pq in pairs:
        o = ORIG["factcheckgpt"].get(tuple(pq)) or ORIG["wikibio"][tuple(pq)]
        for k, lab in enumerate(o.split("-")):
            v = r.dirichlet([1, 1, 1])
            if r.random() < acc: v = 0.2 * v; v[NAMES.index(lab)] += 0.8
            (F if k == 0 else B).append(v.tolist())
    return F, B

# ---- NLI専用のモデル ----
def run_nli(mid, prem, hyp, bs=64):
    tok = AutoTokenizer.from_pretrained(mid)
    m = AutoModelForSequenceClassification.from_pretrained(mid, torch_dtype=torch.float32).cuda().eval()
    idx = []
    for name in NAMES:
        hit = [i for i, l in m.config.id2label.items() if l.lower().startswith(name[:6])]
        assert len(hit) == 1, (mid, m.config.id2label); idx.append(int(hit[0]))
    out, t = [], time.time()
    for i in range(0, len(prem), bs):
        enc = tok(prem[i:i + bs], hyp[i:i + bs], return_tensors="pt", padding=True, truncation=True, max_length=256).to("cuda")
        with torch.no_grad(): p = m(**enc).logits.float().softmax(-1)[:, idx]
        out += p.cpu().tolist()
        if (i // bs) % 50 == 0: print(f"  {i + bs}/{len(prem)}  {time.time() - t:.0f}秒", flush=True)
    del m; torch.cuda.empty_cache()
    return out

P = [p for p, c in RUN]; C = [c for p, c in RUN]
for name, mid in NLI_MODELS.items():
    if name in res["models"]: print("済み:", name); continue
    print(name, mid)
    if USE_MOCK_NLI: F, B = mock(RUN, len(res["models"]), [.6, .75, .9][len(res["models"]) % 3])
    else:
        both = run_nli(mid, P + C, C + P); F, B = both[:len(P)], both[len(P):]   # 親→子 と 子→親
    res["models"][name] = {"fwd": F, "bwd": B}; save()

# ---- Gemma-2 2B に3択で聞く（聞き方3つから、評価に使わない200組で1つ選ぶ）----
PROMPTS = {
 "P1_前提と仮説": ("Premise: {a}\nHypothesis: {b}\nDoes the premise entail the hypothesis?\n"
                "A. Yes, the hypothesis must be true.\nB. Cannot tell.\nC. No, the hypothesis must be false.\n"
                "Answer with A, B, or C only.", "ABC"),
 "P2_もし本当なら": ("Statement 1: {a}\nStatement 2: {b}\nIf Statement 1 is true, what about Statement 2?\n"
                "A. It must also be true.\nB. It could be true or false.\nC. It must be false.\n"
                "Answer with A, B, or C only.", "ABC"),
 "P3_順番を逆に": ("Statement 1: {a}\nStatement 2: {b}\nIf Statement 1 is true, what about Statement 2?\n"
                "A. It must be false.\nB. It could be true or false.\nC. It must also be true.\n"
                "Answer with A, B, or C only.", "CBA"),   # A=矛盾, C=含まれる（位置の偏りを見る）
}
GNAME = "Gemma-2 2B(聞く)"
def balanced_agree(F, B, pairs):   # 元の判定との一致（3つの答えごとの当たり率の平均）
    hit = collections.defaultdict(list)
    for f, b, pq in zip(F, B, pairs):
        o = ORIG["wikibio"][tuple(pq)].split("-")
        for v, lab in ((f, o[0]), (b, o[1])): hit[lab].append(NAMES[int(np.argmax(v))] == lab)
    return float(np.mean([np.mean(v) for v in hit.values()])), {k: round(float(np.mean(v)), 3) for k, v in hit.items()}

if GNAME in res["models"]: print("済み:", GNAME)
else:
    if USE_MOCK_NLI:
        def ask(key, a, b):
            n = len(a) // 2; F, B = mock(list(zip(a[:n], b[:n])), 7 + len(key), .7); return F + B
    else:
        from transformers import AutoModelForCausalLM
        from google.colab import userdata
        gtok = AutoTokenizer.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), padding_side="left")
        gm = AutoModelForCausalLM.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), torch_dtype=torch.float16, device_map="cuda").eval()
        LET = [gtok.encode(x, add_special_tokens=False)[0] for x in "ABC"]
        def ask(key, a, b, bs=32):
            tmpl, order = PROMPTS[key]
            Q = [gtok.apply_chat_template([{"role": "user", "content": tmpl.format(a=x, b=y)}], tokenize=False, add_generation_prompt=True) for x, y in zip(a, b)]
            out, t = [], time.time()
            for i in range(0, len(Q), bs):
                enc = gtok(Q[i:i + bs], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
                with torch.no_grad(): p = gm(**enc).logits[:, -1, LET].float().softmax(-1).cpu().numpy()
                # 並べ替えて [含まれる, どちらとも, 矛盾] に
                for row in p: out.append([float(row[order.index(L)]) for L in "ABC"])
                if (i // bs) % 50 == 0: print(f"  {i + bs}/{len(Q)}  {time.time() - t:.0f}秒", flush=True)
            return out
    hp = [p for p, c in HELD]; hc = [c for p, c in HELD]
    score = {}
    for key in PROMPTS:
        both = ask(key, hp + hc, hc + hp)
        score[key] = balanced_agree(both[:200], both[200:], HELD); print(key, score[key])
    best = max(score, key=lambda k: score[k][0]); print("選んだ聞き方:", best)
    both = ask(best, P + C, C + P)
    res["models"][GNAME] = {"fwd": both[:len(P)], "bwd": both[len(P):]}
    res["gemma_prompt"] = {"chosen": best, "held_out_scores": score}
    save()

# ---- ざっと確かめる：元の判定とのずれ ----
for name, v in res["models"].items():
    agree = n = 0
    for f, b, pq in zip(v["fwd"], v["bwd"], res["pairs"]):
        o = ORIG["factcheckgpt"].get(tuple(pq)) or ORIG["wikibio"][tuple(pq)]
        agree += f"{NAMES[int(np.argmax(f))]}-{NAMES[int(np.argmax(b))]}" == o; n += 1
    print(f"{name:24s} 元の判定と同じ {agree / n:.1%}")

## 4. 計算（BTProp・BPを、つながりを替えて比べる）

In [ ]:
nli_json = f"{OUT}/E2-12_nli.json"
for ds in ["factcheckgpt", "wikibio"]:
    subprocess.run(f"python /content/e212.py {ds} '{nli_json}' '{OUT}/E212_{ds}.json' | tail -8", shell=True, check=True)

## 5. ここまでで終わり。E2_results の E2-12_nli.json・E212_factcheckgpt.json・E212_wikibio.json をこちらで読みます。